# Dim Theme Hierarchy — Gold Layer

Flattens the self-referencing theme hierarchy from `lego_brickbase.silver.foundation_themes` into a denormalized dimension table.

## What this notebook does

1. **Read** — Loads all records from the silver `foundation_themes` table.
2. **Transform** — Recursively resolves `parent_id` to produce `parent_theme_name`, `root_theme_name`, and `hierarchy_depth`.
3. **Write** — Overwrites the Delta table at `/Volumes/lego_brickbase/gold/delta_volume/dim_theme_hierarchy`.
4. **Register** — Creates the catalog table `lego_brickbase.gold.dim_theme_hierarchy`.

## Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, coalesce


SILVER_TABLE     = "lego_brickbase.silver.foundation_themes"
DELTA_TABLE_PATH = "/Volumes/lego_brickbase/gold/delta_volume/dim_theme_hierarchy"
CATALOG_TABLE    = "lego_brickbase.gold.dim_theme_hierarchy"

## Read and Transform

Iteratively resolve the theme hierarchy by joining each level back to the themes table on `parent_id`. Produces `parent_theme_name`, `root_theme_name`, and `hierarchy_depth`.

In [ ]:
df_themes = spark.table(SILVER_TABLE).select(
    col("theme_key"),
    col("theme_id"),
    col("theme_name"),
    col("parent_id"),
)

# Build a lookup for id -> name
df_lookup = df_themes.select(
    col("theme_id").alias("_lookup_id"),
    col("theme_name").alias("_lookup_name"),
    col("parent_id").alias("_lookup_parent_id"),
)

# Join to get parent theme name
df_with_parent = df_themes.join(
    df_lookup,
    df_themes["parent_id"] == df_lookup["_lookup_id"],
    "left",
).select(
    df_themes["theme_key"],
    df_themes["theme_id"],
    df_themes["theme_name"],
    df_themes["parent_id"],
    col("_lookup_name").alias("parent_theme_name"),
    col("_lookup_parent_id").alias("grandparent_id"),
)

# Iteratively resolve root theme (walk up to 10 levels)
df_root = df_themes.select(
    col("theme_id").alias("_root_id"),
    col("theme_name").alias("_current_name"),
    col("parent_id").alias("_current_parent"),
    lit(0).alias("_depth"),
)

for i in range(10):
    df_next = df_root.filter(col("_current_parent").isNotNull()).join(
        df_lookup,
        col("_current_parent") == col("_lookup_id"),
        "inner",
    ).select(
        col("_root_id"),
        col("_lookup_name").alias("_current_name"),
        col("_lookup_parent_id").alias("_current_parent"),
        (col("_depth") + 1).alias("_depth"),
    )
    df_at_root = df_root.filter(col("_current_parent").isNull())
    df_root = df_at_root.unionByName(df_next)
    if df_next.isEmpty():
        break

df_root_resolved = df_root.select(
    col("_root_id").alias("_resolved_id"),
    col("_current_name").alias("root_theme_name"),
    col("_depth").alias("hierarchy_depth"),
)

df_dim = df_with_parent.join(
    df_root_resolved,
    df_with_parent["theme_id"] == df_root_resolved["_resolved_id"],
    "left",
).select(
    col("theme_key"),
    col("theme_id"),
    col("theme_name"),
    col("parent_id").alias("parent_theme_id"),
    coalesce(col("parent_theme_name"), lit(None)).alias("parent_theme_name"),
    col("root_theme_name"),
    col("hierarchy_depth"),
)

df_dim.printSchema()
df_dim.display(10, truncate=False)

## Write to Delta Volume, Register Catalog Table, and Apply Constraints

In [ ]:
(
    df_dim
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_TABLE_PATH)
)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
print(f"Rows written to Delta table: {row_count:,}")

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG_TABLE} (
        theme_key         INTEGER NOT NULL COMMENT 'Surrogate key for the theme dimension',
        theme_id          INTEGER          COMMENT 'Original theme identifier from the Rebrickable source system',
        theme_name        STRING           COMMENT 'Name of this theme (e.g. Star Wars, Technic, City)',
        parent_theme_id   INTEGER          COMMENT 'theme_id of the immediate parent theme; NULL for root-level themes',
        parent_theme_name STRING           COMMENT 'Name of the immediate parent theme; NULL for root-level themes',
        root_theme_name   STRING           COMMENT 'Name of the top-level ancestor theme; equals theme_name for root-level themes',
        hierarchy_depth   INTEGER          COMMENT 'Depth of this theme in the hierarchy (0 = root, 1 = one level below root, etc.)',
        CONSTRAINT dim_theme_hierarchy_pk PRIMARY KEY (theme_key)
    )
    COMMENT 'Every LEGO theme with its full hierarchy resolved — parent theme name, top-level root theme, and depth level — making it easy to group and filter sets by theme family.'
""")

spark.sql(f"""
    INSERT INTO {CATALOG_TABLE}
    SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")

print(f"Catalog table ready with constraints: {CATALOG_TABLE}")